# Crystal embedding projections

Compute CrystalNN and AMD structural embeddings, fit PCA and UMAP on training
structures only, and transform generated structures into fixed train-defined
spaces. Compare generated structures with training structures using embedding
distances, projected densities, and neighborhood-preservation diagnostics.


In [ ]:
# ruff: noqa: E402, I001
from __future__ import annotations

import gzip
import pickle
import sys
import warnings
from datetime import UTC, datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import Markdown, display
from plotly.colors import hex_to_rgb
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.neighbors import KernelDensity
from pymatgen.core import Composition
from pymatviz import structure_2d
from sklearn.metrics import pairwise_distances_chunked
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

try:
    import amd
    import umap
    from amd.io import periodicset_from_pymatgen_structure
    from matminer.featurizers.site.fingerprint import CrystalNNFingerprint
except ModuleNotFoundError as exc:
    msg = "Install notebook dependencies with `rtk uv sync --group dev`."
    raise ModuleNotFoundError(msg) from exc

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    notebooks_dir = path / "notebooks"
    if (notebooks_dir / "notebook_utils.py").exists():
        if str(notebooks_dir) not in sys.path:
            sys.path.insert(0, str(notebooks_dir))
        break
else:
    raise RuntimeError("Could not find notebooks directory")

from notebook_constants import (  # noqa: E402
    CATEGORY_LABELS,
    CATEGORY_ORDER,
    CRYSTAL_SYSTEM_ORDER,
    WYCKOFF_REPR_FILE,
)
from notebook_utils import (  # noqa: E402
    classify_model,
    crystal_system_from_spg_num,
    find_repo_root,
    load_pickle_gz,
    load_wyckoff_data,
    missing_required_paths,
    required_paths,
)
from plot_style import GRAY, PALETTE  # noqa: E402
from src.config import ANALYSIS_RESULTS_DIR, INPUT_DIR, RAW_RESULTS_DIR  # noqa: E402

ROOT = find_repo_root()
NOTEBOOKS_DIR = ROOT / "notebooks"
CACHE_DIR = NOTEBOOKS_DIR / ".cache" / "embedding_projections"
FIGURE_DIR = ANALYSIS_RESULTS_DIR / "embedding_projections"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

for import_path in (ROOT, NOTEBOOKS_DIR):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

ROOT, INPUT_DIR, RAW_RESULTS_DIR

In [ ]:
UMAP_RANDOM_STATE = 0
SAMPLE_RANDOM_STATE = 0

# Set to None to test all discovered models, or provide model directory names.
MODELS: list[str] | None = None


# Leave as None for the requested full-data analysis. Set to a small integer
# while developing notebook changes.
MAX_STRUCTURES_PER_SPLIT: int | None = None

# Filter generated rows after embedding computation and before projection fitting.
FILTER_GENERATED_TO_METASTABLE_SMACT_VALID = True

# Cap generated samples per novelty category in scatter plots. Train rows always
# define the projection fit uncapped, and likelihood plots use all filtered
# generated samples.
MAX_PLOT_POINTS_PER_GENERATED_CATEGORY: int | None = 50

# Cross section through one existing projection density plot.
CROSS_SECTION_MODEL = "mattergen"
CROSS_SECTION_EMBEDDING = "amd"
CROSS_SECTION_REDUCER = "umap"
CROSS_SECTION_FIXED_COMPONENT = 1
CROSS_SECTION_VALUE = 2.3

# Train KDE is fit in each 2D projection space and generated samples are
# scored with log-likelihood for numerical stability.
KDE_MIN_TRAIN_POINTS = 5
KDE_SCOTT_FACTOR = 0.5

MODEL_PATH_KEYS = (
    "generated_structures",
    "relaxed_ehull",
    "smact_validity",
    "direct_sm",
    "relaxed_sm_anon_matches",
    "relaxed_wyckoff_matches",
    "wyckoff_repr",
)
LABEL_ORDER = ["train data", *(CATEGORY_LABELS[key] for key in CATEGORY_ORDER)]
LABEL_COLORS = {
    "train data": GRAY,
    "Duplicate": PALETTE[0],
    "Substituted (both)": PALETTE[1],
    "Substituted (SM-anon only)": PALETTE[2],
    "Substituted (Wyckoff only)": PALETTE[3],
    "Unmatched": PALETTE[4],
}
CATEGORY_LABEL_ALIASES = {
    "exact match": "Duplicate",
    "subst match: both": "Substituted (both)",
    "subst match: sm-anon": "Substituted (SM-anon only)",
    "subst match: wyckoff": "Substituted (Wyckoff only)",
    "no match": "Unmatched",
    "Substituted (Both)": "Substituted (both)",
    "Substituted (Lattice & site only)": "Substituted (SM-anon only)",
}


def normalized_category_label(label: str) -> str:
    return CATEGORY_LABEL_ALIASES.get(label, label)


SUBST_MATCH_LABELS = {
    "Substituted (both)",
    "Substituted (SM-anon only)",
    "Substituted (Wyckoff only)",
}
MERGED_SUBST_LABEL = "Substituted"
MERGED_LABEL_ORDER = ["train data", "Duplicate", MERGED_SUBST_LABEL, "Unmatched"]
MERGED_LABEL_COLORS = {
    "train data": LABEL_COLORS["train data"],
    "Duplicate": LABEL_COLORS["Duplicate"],
    MERGED_SUBST_LABEL: PALETTE[1],
    "Unmatched": LABEL_COLORS["Unmatched"],
}
PLOTLY_CRYSTAL_SYSTEM_MARKERS = {
    "triclinic": "circle",
    "monoclinic": "square",
    "orthorhombic": "diamond",
    "tetragonal": "cross",
    "trigonal": "x",
    "hexagonal": "triangle-up",
    "cubic": "star",
}
MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS = {
    "triclinic": "o",
    "monoclinic": "s",
    "orthorhombic": "D",
    "tetragonal": "P",
    "trigonal": "X",
    "hexagonal": "^",
    "cubic": "*",
}
EMBEDDING_SETTINGS = {
    "crystalnn": {"preset": "ops"},
    "amd": {"k": 100, "remove_hydrogens": False, "skip_disorder": False},
}
PROJECTION_REDUCERS = ("pca", "umap")
REDUCER_SETTINGS = {
    "pca": {"n_components": 2},
    "umap": {"n_components": 2, "random_state": UMAP_RANDOM_STATE},
}

## Models and labels

Models missing relaxed substitution match artifacts are skipped because the requested six legend classes cannot be derived for them.

In [ ]:
def discover_models() -> list[str]:
    gen_dir = INPUT_DIR / "gen" / "preprocessed"
    return sorted(
        path.name
        for path in gen_dir.iterdir()
        if path.is_dir() and not path.name.startswith(".")
    )


def select_models(
    discovered_models: list[str],
    requested_models: list[str] | None,
) -> list[str]:
    if requested_models is None:
        return discovered_models
    if not requested_models:
        raise ValueError("MODELS must be None or contain at least one model.")

    seen = set()
    duplicate_models = []
    for model in requested_models:
        if model in seen and model not in duplicate_models:
            duplicate_models.append(model)
        seen.add(model)
    if duplicate_models:
        raise ValueError(f"MODELS contains duplicates: {duplicate_models}")

    unknown_models = [
        model for model in requested_models if model not in discovered_models
    ]
    if unknown_models:
        raise ValueError(f"MODELS contains unknown models: {unknown_models}")
    return list(requested_models)


def model_required_paths(model: str) -> dict[str, Path]:
    return required_paths(model, INPUT_DIR, RAW_RESULTS_DIR, MODEL_PATH_KEYS)


discovered_models = discover_models()
models = select_models(discovered_models, MODELS)
missing_files = pd.DataFrame(
    record
    for model in models
    for record in missing_required_paths(model_required_paths(model), model=model)
)
skipped_models = (
    sorted(missing_files["model"].unique()) if not missing_files.empty else []
)
if MODELS is not None and skipped_models:
    missing_by_model = (
        missing_files.groupby("model", sort=False)["kind"].agg(list).to_dict()
    )
    details = "; ".join(
        f"{model}: {', '.join(missing_by_model[model])}"
        for model in models
        if model in missing_by_model
    )
    raise ValueError(f"Requested models are missing required files: {details}")
complete_models = [model for model in models if model not in skipped_models]

if skipped_models:
    display(Markdown(f"Skipped incomplete models: `{', '.join(skipped_models)}`"))
    display(missing_files)

classifications = pd.concat(
    [
        classify_model(
            model,
            model_required_paths(model),
            include_category_label=True,
            include_crystal_system=True,
        )
        for model in complete_models
    ],
    ignore_index=True,
)

if not classifications.empty:
    classifications["space_group"] = classifications["gen_idx"].map(int)
    for model in complete_models:
        paths = model_required_paths(model)
        wyckoff_data = load_pickle_gz(paths["wyckoff_repr"])
        mask = classifications["model"] == model
        gen_indices = classifications.loc[mask, "gen_idx"].astype(int)
        classifications.loc[mask, "space_group"] = [
            int(wyckoff_data[gen_idx].spg_num) for gen_idx in gen_indices
        ]

    classifications["space_group"] = classifications["space_group"].astype(int)

display(classifications.groupby(["model", "category_label"]).size())
complete_models

## Cache helpers

In [ ]:
def load_cache(path: Path) -> Any | None:
    if not path.exists():
        return None
    with gzip.open(path, "rb") as file:
        return pickle.load(file)  # noqa: S301


def save_cache(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wb") as file:
        pickle.dump(payload, file, protocol=pickle.HIGHEST_PROTOCOL)


def migrate_embedding_cache_payload(
    payload: dict[str, Any],
) -> tuple[dict[str, Any], bool]:
    embeddings = payload["embeddings"]
    if isinstance(embeddings, list) and "successful_indices" in payload:
        return payload, False

    embeddings = np.asarray(embeddings, dtype=float)
    migrated = dict(payload)
    n_samples = len(embeddings)
    migrated["embeddings"] = embeddings.tolist()
    migrated["successful_indices"] = list(range(n_samples))
    migrated["failed_indices"] = []
    migrated["failures"] = []
    return migrated, True


def migrate_embedding_caches(cache_dir: Path) -> int:
    migrated_count = 0
    if not cache_dir.exists():
        return migrated_count

    for path in sorted(cache_dir.glob("*.pkl.gz")):
        payload = load_cache(path)
        if payload is None:
            continue
        migrated, changed = migrate_embedding_cache_payload(payload)
        if changed:
            save_cache(path, migrated)
            migrated_count += 1
    return migrated_count


migrated_embedding_cache_count = migrate_embedding_caches(CACHE_DIR / "embeddings")
if migrated_embedding_cache_count:
    display(
        Markdown(
            f"Migrated {migrated_embedding_cache_count} embedding cache files "
            "to failure-aware format."
        )
    )


def train_embedding_cache_path(embedding_name: str, n_samples: int) -> Path:
    name = f"train_{embedding_name}_{n_samples}.pkl.gz"
    return CACHE_DIR / "embeddings" / name


def generated_embedding_cache_path(
    model: str,
    embedding_name: str,
    n_samples: int,
) -> Path:
    name = f"{model}_{embedding_name}_{n_samples}.pkl.gz"
    return CACHE_DIR / "embeddings" / name


def projection_seed_label(reducer: str, generated_cap: int | None) -> str:
    labels = []
    if reducer == "umap":
        labels.append(f"umap_seed={UMAP_RANDOM_STATE}")
    if generated_cap is not None:
        labels.append(f"sample_seed={SAMPLE_RANDOM_STATE}")
    return "".join(f"_{label}" for label in labels)


def projection_cache_path(
    model: str,
    embedding_name: str,
    reducer: str,
    selection: str,
    generated_cap: int | None,
) -> Path:
    metric = embedding_distance_metric(embedding_name) if reducer == "umap" else None
    metric_label = f"_metric={metric}" if metric is not None else ""
    cap_label = "all" if generated_cap is None else str(generated_cap)
    seed_label = projection_seed_label(reducer, generated_cap)
    name = (
        f"train_fit_{model}_{embedding_name}_{reducer}"
        f"_{selection}"
        f"{metric_label}"
        f"_f={FILTER_GENERATED_TO_METASTABLE_SMACT_VALID}"
        f"_n={cap_label}"
        f"{seed_label}"
        ".pkl.gz"
    )
    return CACHE_DIR / "projections" / name


def distance_matrix_cache_path(
    model: str,
    embedding_name: str,
    n_generated: int,
    n_train: int,
) -> Path:
    name = f"{model}_{embedding_name}_gen={n_generated}_train={n_train}.pkl.gz"
    return CACHE_DIR / "distance_matrices" / name

## Embeddings

CrystalNN is a site-level featurizer, so each structure is embedded by featurizing every site and averaging the site vectors. AMD is a structure-level geometric descriptor computed from each pymatgen structure converted to an AMD periodic set.


In [ ]:
def maybe_limit_structures(structures: list[Any]) -> list[Any]:
    if MAX_STRUCTURES_PER_SPLIT is None:
        return structures
    return structures[:MAX_STRUCTURES_PER_SPLIT]


def make_featurizer(embedding_name: str) -> Any | None:
    if embedding_name == "crystalnn":
        return CrystalNNFingerprint.from_preset("ops")
    if embedding_name == "amd":
        return None
    raise ValueError(f"Unknown embedding: {embedding_name}")


def crystalnn_embedding(
    structure: Any,
    featurizer: Any,
    *,
    structure_idx: int,
) -> np.ndarray:
    site_features = []
    for site_idx in range(len(structure)):
        try:
            values = np.asarray(featurizer.featurize(structure, site_idx), dtype=float)
        except Exception as exc:
            msg = f"crystalnn failed for structure {structure_idx}, site {site_idx}."
            raise RuntimeError(msg) from exc
        site_features.append(values)

    embedding = np.vstack(site_features).mean(axis=0)
    if not np.all(np.isfinite(embedding)):
        raise ValueError(
            f"crystalnn produced non-finite values for structure {structure_idx}."
        )
    return embedding


def amd_embedding(structure: Any, *, structure_idx: int) -> np.ndarray:
    settings = EMBEDDING_SETTINGS["amd"]
    try:
        periodic_set = periodicset_from_pymatgen_structure(
            structure,
            remove_hydrogens=settings["remove_hydrogens"],
            skip_disorder=settings["skip_disorder"],
        )
        embedding = np.asarray(amd.AMD(periodic_set, settings["k"]), dtype=float)
    except Exception as exc:
        msg = f"amd failed for structure {structure_idx}."
        raise RuntimeError(msg) from exc

    expected_shape = (settings["k"],)
    if embedding.shape != expected_shape:
        raise ValueError(
            f"amd produced shape {embedding.shape} for structure {structure_idx}; "
            f"expected {expected_shape}."
        )
    if not np.all(np.isfinite(embedding)):
        raise ValueError(
            f"amd produced non-finite values for structure {structure_idx}."
        )
    return embedding


def structure_embedding(
    structure: Any,
    featurizer: Any | None,
    *,
    structure_idx: int,
    embedding_name: str,
) -> np.ndarray:
    if embedding_name == "crystalnn":
        return crystalnn_embedding(
            structure,
            featurizer,
            structure_idx=structure_idx,
        )
    if embedding_name == "amd":
        return amd_embedding(structure, structure_idx=structure_idx)
    raise ValueError(f"Unknown embedding: {embedding_name}")


def embedding_failure_record(structure_idx: int, exc: Exception) -> dict[str, Any]:
    return {
        "structure_idx": structure_idx,
        "exception_type": type(exc).__name__,
        "message": str(exc),
    }


def compute_embeddings(
    structures: list[Any],
    featurizer: Any | None,
    *,
    embedding_name: str,
    desc: str,
    allow_failures: bool,
) -> dict[str, Any]:
    embeddings: list[list[float] | None] = []
    successful_indices: list[int] = []
    failed_indices: list[int] = []
    failures: list[dict[str, Any]] = []

    for structure_idx, structure in enumerate(tqdm(structures, desc=desc)):
        try:
            embedding = structure_embedding(
                structure,
                featurizer,
                structure_idx=structure_idx,
                embedding_name=embedding_name,
            )
        except Exception as exc:
            if not allow_failures:
                raise
            embeddings.append(None)
            failed_indices.append(structure_idx)
            failures.append(embedding_failure_record(structure_idx, exc))
            continue

        embeddings.append(embedding.tolist())
        successful_indices.append(structure_idx)

    if failures:
        warnings.warn(
            f"{desc}: skipped {len(failures)} structures with failed embeddings.",
            stacklevel=2,
        )

    return {
        "embeddings": embeddings,
        "successful_indices": successful_indices,
        "failed_indices": failed_indices,
        "failures": failures,
    }


def successful_embedding_array(
    payload: dict[str, Any],
    *,
    embedding_size: int | None = None,
) -> np.ndarray:
    embeddings = payload["embeddings"]
    successful_indices = payload["successful_indices"]
    rows = [embeddings[idx] for idx in successful_indices]
    if rows:
        return np.asarray(rows, dtype=float)
    if embedding_size is None:
        raise ValueError("Embedding payload has no successful rows.")
    return np.empty((0, embedding_size), dtype=float)


def load_or_compute_embeddings(
    *,
    cache_path: Path,
    embedding_name: str,
    structures: list[Any],
    featurizer: Any | None,
    desc: str,
    metadata: dict[str, Any],
    allow_failures: bool,
) -> dict[str, Any]:
    cached = load_cache(cache_path)
    if cached is not None:
        return cached

    payload = compute_embeddings(
        structures,
        featurizer,
        embedding_name=embedding_name,
        desc=desc,
        allow_failures=allow_failures,
    )
    payload = {
        "metadata": metadata,
        "created_at": datetime.now(UTC).isoformat(),
        **payload,
    }
    save_cache(cache_path, payload)
    return payload

In [ ]:
train_path = INPUT_DIR / "train" / "preprocessed" / "train.pkl.gz"
train_structures = maybe_limit_structures(load_pickle_gz(train_path))

train_embedding_payloads_by_name: dict[str, dict[str, Any]] = {}
train_embeddings_by_name: dict[str, np.ndarray] = {}
for embedding_name in EMBEDDING_SETTINGS:
    featurizer = make_featurizer(embedding_name)
    n_samples = len(train_structures)
    payload = load_or_compute_embeddings(
        cache_path=train_embedding_cache_path(embedding_name, n_samples),
        embedding_name=embedding_name,
        structures=train_structures,
        featurizer=featurizer,
        desc=f"train {embedding_name}",
        metadata={
            "split": "train",
            "embedding_name": embedding_name,
            "n_samples": n_samples,
        },
        allow_failures=False,
    )
    if payload["failed_indices"]:
        raise ValueError(
            f"Train {embedding_name} cache contains failed embeddings: "
            f"{payload['failed_indices'][:5]}"
        )
    train_embedding_payloads_by_name[embedding_name] = payload
    train_embeddings_by_name[embedding_name] = successful_embedding_array(payload)

embedding_tables: dict[tuple[str, str], pd.DataFrame] = {}
embedding_arrays: dict[tuple[str, str], np.ndarray] = {}
embedding_failures: dict[tuple[str, str], list[dict[str, Any]]] = {}
for model in complete_models:
    generated_path = model_required_paths(model)["generated_structures"]
    generated_structures = maybe_limit_structures(load_pickle_gz(generated_path))
    model_labels = classifications[classifications["model"] == model].reset_index(
        drop=True
    )
    if MAX_STRUCTURES_PER_SPLIT is not None:
        model_labels = model_labels.iloc[: len(generated_structures)].reset_index(
            drop=True
        )

    generated_compositions = np.array(
        [structure.composition.reduced_formula for structure in generated_structures],
        dtype=object,
    )

    for embedding_name in EMBEDDING_SETTINGS:
        featurizer = make_featurizer(embedding_name)
        train_embeddings = train_embeddings_by_name[embedding_name]
        n_samples = len(generated_structures)

        generated_payload = load_or_compute_embeddings(
            cache_path=generated_embedding_cache_path(
                model,
                embedding_name,
                n_samples,
            ),
            embedding_name=embedding_name,
            structures=generated_structures,
            featurizer=featurizer,
            desc=f"{model} generated {embedding_name}",
            metadata={
                "model": model,
                "split": "generated",
                "embedding_name": embedding_name,
                "n_samples": n_samples,
            },
            allow_failures=True,
        )
        generated_embeddings = successful_embedding_array(
            generated_payload,
            embedding_size=train_embeddings.shape[1],
        )
        successful_indices = np.asarray(
            generated_payload["successful_indices"],
            dtype=int,
        )
        failures = list(generated_payload["failures"])
        embedding_failures[(model, embedding_name)] = failures
        if failures:
            display(
                Markdown(
                    f"`{model}` `{embedding_name}`: skipped {len(failures)} "
                    "structures with failed embeddings."
                )
            )

        successful_labels = model_labels.iloc[successful_indices].reset_index(drop=True)

        metadata = pd.concat(
            [
                pd.DataFrame(
                    {
                        "model": model,
                        "split": "train",
                        "structure_idx": np.arange(len(train_embeddings)),
                        "gen_idx": pd.NA,
                        "label": "train data",
                        "composition": pd.NA,
                        "crystal_system": pd.NA,
                        "space_group": pd.NA,
                        "is_metastable_smact_valid": pd.NA,
                        "embedding_successful": True,
                    }
                ),
                pd.DataFrame(
                    {
                        "model": model,
                        "split": "generated",
                        "structure_idx": successful_labels["gen_idx"].to_numpy(),
                        "gen_idx": successful_labels["gen_idx"].to_numpy(),
                        "label": successful_labels["category_label"].astype(str),
                        "composition": generated_compositions[successful_indices],
                        "crystal_system": successful_labels["crystal_system"].astype(
                            str
                        ),
                        "space_group": successful_labels["space_group"].astype(int),
                        "is_metastable_smact_valid": successful_labels[
                            "is_metastable_smact_valid"
                        ].astype(bool),
                        "embedding_successful": True,
                    }
                ),
            ],
            ignore_index=True,
        )
        embedding_arrays[(model, embedding_name)] = np.vstack(
            [train_embeddings, generated_embeddings]
        )
        embedding_tables[(model, embedding_name)] = metadata

        display(
            Markdown(
                f"`{model}` `{embedding_name}`: "
                f"{embedding_arrays[(model, embedding_name)].shape}"
            )
        )

## Minimum generated-to-training embedding distances

Compare generated structures with training structures in the original embedding
space. CrystalNN uses Euclidean distance and AMD uses Chebyshev distance.


In [ ]:
def embedding_distance_metric(embedding_name: str) -> str:
    if embedding_name == "crystalnn":
        return "euclidean"
    if embedding_name == "amd":
        return "chebyshev"
    raise ValueError(f"Unknown embedding: {embedding_name}")


def validate_distance_matrix(
    values: np.ndarray,
    *,
    expected_shape: tuple[int, int],
    path: Path,
) -> np.ndarray:
    if values.shape != expected_shape:
        raise ValueError(
            f"{path} has unexpected shape {values.shape}; expected {expected_shape}."
        )
    if not np.all(np.isfinite(values)):
        raise ValueError(f"{path} contains non-finite distances.")
    return values


def compute_distance_matrix_to_train(
    *,
    generated_embeddings: np.ndarray,
    train_embeddings: np.ndarray,
    embedding_name: str,
) -> np.ndarray:
    metric = embedding_distance_metric(embedding_name)
    chunks = []
    for chunk in pairwise_distances_chunked(
        generated_embeddings,
        train_embeddings,
        metric=metric,
    ):
        chunks.append(np.asarray(chunk, dtype=np.float32))
    return np.vstack(chunks)


def load_or_compute_distance_matrix(
    *,
    cache_path: Path,
    model: str,
    embedding_name: str,
    generated_embeddings: np.ndarray,
    train_embeddings: np.ndarray,
) -> np.ndarray:
    expected_shape = (len(generated_embeddings), len(train_embeddings))
    cached = load_cache(cache_path)
    if cached is not None:
        values = np.asarray(cached["distances"], dtype=np.float32)
        return validate_distance_matrix(
            values,
            expected_shape=expected_shape,
            path=cache_path,
        )

    distances = compute_distance_matrix_to_train(
        generated_embeddings=generated_embeddings,
        train_embeddings=train_embeddings,
        embedding_name=embedding_name,
    )
    payload = {
        "metadata": {
            "model": model,
            "embedding_name": embedding_name,
            "metric": embedding_distance_metric(embedding_name),
            "n_generated": len(generated_embeddings),
            "n_train": len(train_embeddings),
            "dtype": "float32",
        },
        "created_at": datetime.now(UTC).isoformat(),
        "distances": distances,
    }
    save_cache(cache_path, payload)
    return distances


distance_matrices: dict[tuple[str, str], np.ndarray] = {}


def distance_matrix(model: str, embedding_name: str) -> np.ndarray:
    key = (model, embedding_name)
    if key in distance_matrices:
        return distance_matrices[key]

    train_embeddings = train_embeddings_by_name[embedding_name]
    n_train = len(train_embeddings)
    generated_embeddings = embedding_arrays[key][n_train:]
    cache_path = distance_matrix_cache_path(
        model,
        embedding_name,
        len(generated_embeddings),
        n_train,
    )
    distances = load_or_compute_distance_matrix(
        cache_path=cache_path,
        model=model,
        embedding_name=embedding_name,
        generated_embeddings=generated_embeddings,
        train_embeddings=train_embeddings,
    )
    distance_matrices[key] = distances
    return distances


def merged_distance_label(label: str) -> str:
    label = normalized_category_label(label)
    if label in SUBST_MATCH_LABELS:
        return MERGED_SUBST_LABEL
    return label


def wyckoff_composition(wyckoff_data: Any) -> str:
    counts: dict[str, int] = {}
    for element, atom_indices in zip(
        wyckoff_data.orbit_elements,
        wyckoff_data.orbit_atom_indices,
        strict=True,
    ):
        counts[str(element)] = counts.get(str(element), 0) + len(atom_indices)
    return Composition(counts).reduced_formula


def train_metadata_from_wyckoff(path: Path, expected: int) -> pd.DataFrame:
    records = load_wyckoff_data(path, expected)
    return pd.DataFrame(
        [
            {
                "train_idx": idx,
                "composition": wyckoff_composition(record),
                "crystal_system": crystal_system_from_spg_num(int(record.spg_num)),
                "space_group": int(record.spg_num),
            }
            for idx, record in enumerate(records)
        ]
    )


train_metadata = train_metadata_from_wyckoff(
    RAW_RESULTS_DIR / "train" / WYCKOFF_REPR_FILE,
    len(train_structures),
)


def minimum_distance_frame(model: str, embedding_name: str) -> pd.DataFrame:
    metadata = embedding_tables[(model, embedding_name)]
    generated_metadata = metadata[metadata["split"] == "generated"].reset_index(
        drop=True
    )
    distances = distance_matrix(model, embedding_name)
    nearest_train_indices = distances.argmin(axis=1)
    minimum_distances = distances[np.arange(len(distances)), nearest_train_indices]
    return pd.DataFrame(
        {
            "model": model,
            "embedding_name": embedding_name,
            "gen_idx": generated_metadata["gen_idx"].astype(int),
            "label": generated_metadata["label"].astype(str),
            "merged_label": generated_metadata["label"]
            .astype(str)
            .map(merged_distance_label),
            "composition": generated_metadata["composition"].astype(str),
            "crystal_system": generated_metadata["crystal_system"].astype(str),
            "space_group": generated_metadata["space_group"].astype(int),
            "is_metastable_smact_valid": generated_metadata[
                "is_metastable_smact_valid"
            ].astype(bool),
            "minimum_distance": minimum_distances,
            "nearest_train_idx": nearest_train_indices.astype(int),
        }
    )


def minimum_distance_plot_frame(frame: pd.DataFrame) -> pd.DataFrame:
    return frame[frame["is_metastable_smact_valid"].astype(bool)].reset_index(drop=True)


def kde_display_bound(frame: pd.DataFrame) -> tuple[float, int]:
    if frame.empty:
        return np.nan, 0
    upper = float(frame["minimum_distance"].quantile(0.99))
    clipped_count = int((frame["minimum_distance"] > upper).sum())
    return upper, clipped_count


def draw_minimum_distance_distribution(
    frame: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
    ax: plt.Axes,
) -> tuple[list[Any], list[str]]:
    upper, clipped_count = kde_display_bound(frame)
    labels = [label for label in MERGED_LABEL_ORDER if label != "train data"]
    present_labels = []
    if not frame.empty:
        present_labels = [
            label for label in labels if label in set(frame["merged_label"].astype(str))
        ]

    legend_handles: list[Any] = []
    legend_labels: list[str] = []
    if present_labels:
        sns.kdeplot(
            data=frame,
            x="minimum_distance",
            hue="merged_label",
            hue_order=present_labels,
            palette={label: MERGED_LABEL_COLORS[label] for label in present_labels},
            common_norm=False,
            fill=True,
            alpha=0.25,
            linewidth=1.8,
            cut=0,
            ax=ax,
        )
        kde_legend = ax.get_legend()
        if kde_legend is not None:
            legend_handles = getattr(kde_legend, "legend_handles", None)
            if legend_handles is None:
                legend_handles = getattr(kde_legend, "legendHandles", [])
            legend_labels = [text.get_text() for text in kde_legend.texts]
            kde_legend.remove()
        for label in present_labels:
            mean_distance = frame.loc[
                frame["merged_label"] == label,
                "minimum_distance",
            ].mean()
            ax.axvline(
                mean_distance,
                color=MERGED_LABEL_COLORS[label],
                linestyle="--",
                linewidth=1.4,
                label="_nolegend_",
            )
    ax.set_xlabel(f"{embedding_name} distance to nearest train structure")
    ax.set_ylabel("density")
    ax.set_title(embedding_name)
    if np.isfinite(upper):
        ax.set_xlim(left=0, right=upper)
    if clipped_count:
        ax.text(
            0.98,
            0.96,
            f"clipped: {clipped_count}",
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=8,
        )
    return list(legend_handles), legend_labels


MINIMUM_DISTANCE_EMBEDDING_ORDER = ("crystalnn", "amd")


def plot_minimum_distance_distribution_grid(
    frames: dict[str, pd.DataFrame],
    *,
    model: str,
    path: Path | None = None,
) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), constrained_layout=True)
    legend_handles: list[Any] = []
    legend_labels: list[str] = []
    for ax, embedding_name in zip(
        axes,
        MINIMUM_DISTANCE_EMBEDDING_ORDER,
        strict=True,
    ):
        handles, labels = draw_minimum_distance_distribution(
            frames[embedding_name],
            model=model,
            embedding_name=embedding_name,
            ax=ax,
        )
        if handles and not legend_handles:
            legend_handles = handles
            legend_labels = labels

    if legend_handles:
        fig.legend(
            legend_handles,
            legend_labels,
            title="category",
            loc="lower center",
            ncol=min(len(legend_labels), 3),
            frameon=False,
            bbox_to_anchor=(0.5, -0.04),
        )
    fig.suptitle(f"{model}: distance to nearest train structure")
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
    display(fig)
    plt.close(fig)


minimum_distance_tables: dict[tuple[str, str], pd.DataFrame] = {}
for model in complete_models:
    model_distance_frames: dict[str, pd.DataFrame] = {}
    for embedding_name in MINIMUM_DISTANCE_EMBEDDING_ORDER:
        frame = minimum_distance_frame(model, embedding_name)
        plot_frame = minimum_distance_plot_frame(frame)
        minimum_distance_tables[(model, embedding_name)] = plot_frame
        model_distance_frames[embedding_name] = plot_frame
    plot_minimum_distance_distribution_grid(
        model_distance_frames,
        model=model,
        path=FIGURE_DIR / f"{model}_minimum_train_distances.pdf",
    )
minimum_distance_summary = (
    pd.concat(
        minimum_distance_tables.values(),
        ignore_index=True,
    )
    .groupby(["model", "embedding_name", "merged_label"], observed=False)[
        "minimum_distance"
    ]
    .describe()
)
display(minimum_distance_summary)

## Train-fit projections and plots

Fit scaling, PCA, and UMAP on training structures only, then transform generated
structures into those fixed spaces. All projection-based distance plots and
diagnostics below use these train-fit coordinates.


In [ ]:
def projection_sampling_label(label: str) -> str:
    label = normalized_category_label(label)
    if label in SUBST_MATCH_LABELS:
        return MERGED_SUBST_LABEL
    return label


def sample_generated_category(frame: pd.DataFrame, cap: int | None) -> pd.DataFrame:
    if cap is None:
        return frame

    label = str(frame.name)
    if len(frame) < cap:
        warnings.warn(
            f"Generated category {label!r} has {len(frame)} samples, below cap "
            f"{cap}; plotting all {len(frame)}.",
            stacklevel=2,
        )
        return frame

    return frame.sample(n=cap, random_state=SAMPLE_RANDOM_STATE, replace=False)


def selected_generated_metadata(
    metadata: pd.DataFrame,
    *,
    generated_cap: int | None,
) -> pd.DataFrame:
    generated = metadata[metadata["split"] == "generated"]
    generated = generated[generated["embedding_successful"].astype(bool)]

    if FILTER_GENERATED_TO_METASTABLE_SMACT_VALID:
        generated = generated[generated["is_metastable_smact_valid"].astype(bool)]

    if generated_cap is None:
        return generated

    sampling_labels = generated["label"].astype(str).map(projection_sampling_label)
    return generated.groupby(
        sampling_labels,
        group_keys=False,
        observed=False,
    ).apply(sample_generated_category, cap=generated_cap, include_groups=False)


def selected_projection_indices(
    metadata: pd.DataFrame,
    *,
    generated_cap: int | None,
) -> np.ndarray:
    train = metadata[metadata["split"] == "train"]
    generated = selected_generated_metadata(metadata, generated_cap=generated_cap)
    return np.array(sorted({*train.index, *generated.index}))


def build_reducer(reducer: str, embedding_name: str) -> Any:
    if reducer == "pca":
        return PCA(**REDUCER_SETTINGS["pca"])
    if reducer == "umap":
        settings = dict(REDUCER_SETTINGS["umap"])
        settings["metric"] = embedding_distance_metric(embedding_name)
        return umap.UMAP(**settings)
    raise ValueError(f"Unknown reducer: {reducer}")


def project_embeddings(
    *,
    model: str,
    embedding_name: str,
    reducer: str,
    embeddings: np.ndarray,
    metadata: pd.DataFrame,
    selection: str,
    generated_cap: int | None,
) -> pd.DataFrame:
    indices = selected_projection_indices(metadata, generated_cap=generated_cap)
    selected_metadata = metadata.iloc[indices].reset_index(drop=True)
    cache_path = projection_cache_path(
        model,
        embedding_name,
        reducer,
        selection,
        generated_cap,
    )
    cached = load_cache(cache_path)
    if cached is not None:
        return cached

    train_indices = metadata.index[metadata["split"] == "train"].to_numpy()
    generated_indices = np.array(
        [index for index in indices if metadata.iloc[index]["split"] == "generated"],
        dtype=int,
    )

    scaler = StandardScaler().fit(embeddings[train_indices])
    train_scaled = scaler.transform(embeddings[train_indices])
    reducer_model = build_reducer(reducer, embedding_name).fit(train_scaled)

    train_metadata = metadata.iloc[train_indices].reset_index(drop=True)
    train_coords = reducer_model.transform(train_scaled)
    projection_parts = [
        train_metadata.assign(
            x=train_coords[:, 0],
            y=train_coords[:, 1],
            reducer=reducer,
        )
    ]

    if len(generated_indices):
        generated_metadata = metadata.iloc[generated_indices].reset_index(drop=True)
        generated_scaled = scaler.transform(embeddings[generated_indices])
        generated_coords = reducer_model.transform(generated_scaled)
        projection_parts.append(
            generated_metadata.assign(
                x=generated_coords[:, 0],
                y=generated_coords[:, 1],
                reducer=reducer,
            )
        )

    projections = pd.concat(projection_parts, ignore_index=True)
    projections = projections.merge(
        selected_metadata[["split", "structure_idx", "gen_idx"]].reset_index(
            names="selection_order"
        ),
        on=["split", "structure_idx", "gen_idx"],
        how="left",
        validate="one_to_one",
    ).sort_values("selection_order", kind="stable")
    projections = projections.drop(columns="selection_order").reset_index(drop=True)
    save_cache(cache_path, projections)

    return projections


def projection_frame(
    model: str,
    embedding_name: str,
    *,
    selection: str = "scatter",
    generated_cap: int | None = MAX_PLOT_POINTS_PER_GENERATED_CATEGORY,
) -> pd.DataFrame:
    embeddings = embedding_arrays[(model, embedding_name)]
    metadata = embedding_tables[(model, embedding_name)]
    return pd.concat(
        [
            project_embeddings(
                model=model,
                embedding_name=embedding_name,
                reducer=reducer,
                embeddings=embeddings,
                metadata=metadata,
                selection=selection,
                generated_cap=generated_cap,
            )
            for reducer in PROJECTION_REDUCERS
        ],
        ignore_index=True,
    )


def likelihood_projection_frame(model: str, embedding_name: str) -> pd.DataFrame:
    return projection_frame(
        model,
        embedding_name,
        selection="likelihood_all_filtered",
        generated_cap=None,
    )

In [ ]:
def rgba(color: str, alpha: float) -> str:
    red, green, blue = hex_to_rgb(color)
    return f"rgba({red}, {green}, {blue}, {alpha})"


def reducer_title(reducer: str) -> str:
    return reducer.upper()


def merged_subst_label(label: str) -> str:
    label = normalized_category_label(label)
    if label in SUBST_MATCH_LABELS:
        return MERGED_SUBST_LABEL
    return label


def with_merged_subst_label(projections: pd.DataFrame) -> pd.DataFrame:
    return projections.assign(
        merged_label=projections["label"].astype(str).map(merged_subst_label)
    )


def add_merged_plot_legend(fig: go.Figure) -> None:
    for label in MERGED_LABEL_ORDER:
        if label == "train data":
            continue
        fig.add_trace(
            go.Scattergl(
                x=[None],
                y=[None],
                mode="markers",
                marker={
                    "size": 8,
                    "color": MERGED_LABEL_COLORS[label],
                    "symbol": "circle",
                    "opacity": 0.82,
                    "line": {"width": 0},
                },
                name=label,
                legendgroup=f"category:{label}",
                showlegend=True,
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )

    for crystal_system in CRYSTAL_SYSTEM_ORDER:
        fig.add_trace(
            go.Scattergl(
                x=[None],
                y=[None],
                mode="markers",
                marker={
                    "size": 8,
                    "color": "#333333",
                    "symbol": PLOTLY_CRYSTAL_SYSTEM_MARKERS[crystal_system],
                    "opacity": 0.82,
                    "line": {"width": 0},
                },
                name=crystal_system,
                legendgroup=f"crystal_system:{crystal_system}",
                showlegend=True,
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )


def plot_merged_subst_projection_grid(
    projections: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
) -> go.Figure:
    projections = with_merged_subst_label(projections)
    fig = make_subplots(
        rows=1,
        cols=len(PROJECTION_REDUCERS),
        subplot_titles=[reducer_title(reducer) for reducer in PROJECTION_REDUCERS],
        horizontal_spacing=0.06,
    )

    for col, reducer in enumerate(PROJECTION_REDUCERS, start=1):
        frame = projections[projections["reducer"] == reducer]
        train = frame[frame["merged_label"] == "train data"]
        if not train.empty:
            fig.add_trace(
                go.Histogram2dContour(
                    x=train["x"],
                    y=train["y"],
                    colorscale=[
                        [0, rgba(MERGED_LABEL_COLORS["train data"], 0.0)],
                        [1, rgba(MERGED_LABEL_COLORS["train data"], 0.65)],
                    ],
                    contours={"coloring": "fill", "showlines": True},
                    line={
                        "color": rgba(MERGED_LABEL_COLORS["train data"], 0.8),
                        "width": 1,
                    },
                    opacity=0.55,
                    showscale=False,
                    name="train data density",
                    legendgroup="train data",
                    showlegend=col == 1,
                    hoverinfo="skip",
                ),
                row=1,
                col=col,
            )

        for label in MERGED_LABEL_ORDER:
            if label == "train data":
                continue
            label_frame = frame[frame["merged_label"] == label]
            if label_frame.empty:
                continue
            for crystal_system in CRYSTAL_SYSTEM_ORDER:
                subset = label_frame[label_frame["crystal_system"] == crystal_system]
                if subset.empty:
                    continue
                fig.add_trace(
                    go.Scattergl(
                        x=subset["x"],
                        y=subset["y"],
                        mode="markers",
                        marker={
                            "size": 7,
                            "color": MERGED_LABEL_COLORS[label],
                            "symbol": PLOTLY_CRYSTAL_SYSTEM_MARKERS[crystal_system],
                            "opacity": 0.82,
                            "line": {"width": 0},
                        },
                        name=f"{label}; {crystal_system}",
                        legendgroup=f"category:{label}",
                        showlegend=False,
                        customdata=np.stack(
                            [
                                subset["composition"].astype(str),
                                subset["crystal_system"].astype(str),
                                subset["space_group"].astype(str),
                                subset["gen_idx"].astype(str),
                                subset["merged_label"].astype(str),
                                subset["label"].astype(str),
                                subset["model"].astype(str),
                            ],
                            axis=-1,
                        ),
                        hovertemplate=(
                            "composition: %{customdata[0]}<br>"
                            "crystal system: %{customdata[1]}<br>"
                            "space group: %{customdata[2]}<br>"
                            "gen idx: %{customdata[3]}<br>"
                            "category: %{customdata[4]}<br>"
                            "original category: %{customdata[5]}<br>"
                            "model: %{customdata[6]}"
                            "<extra></extra>"
                        ),
                    ),
                    row=1,
                    col=col,
                )

        fig.update_xaxes(title_text="component 1", row=1, col=col)
        fig.update_yaxes(title_text="component 2", row=1, col=col)

    add_merged_plot_legend(fig)
    title = f"{model}: {embedding_name} embeddings, merged substitution matches"
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1200,
        height=470,
        legend={"orientation": "h", "y": -0.26, "x": 0.5, "xanchor": "center"},
        margin={"l": 55, "r": 20, "t": 70, "b": 120},
    )
    return fig


def save_merged_subst_projection_pdf(
    projections: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
    path: Path,
) -> None:
    projections = with_merged_subst_label(projections)
    fig, axes = plt.subplots(
        1,
        len(PROJECTION_REDUCERS),
        figsize=(10, 5.2),
        constrained_layout=True,
    )
    for ax, reducer in zip(np.ravel(axes), PROJECTION_REDUCERS, strict=True):
        frame = projections[projections["reducer"] == reducer]
        train = frame[frame["merged_label"] == "train data"]
        if not train.empty:
            density, x_edges, y_edges = np.histogram2d(train["x"], train["y"], bins=40)
            x_centers = (x_edges[:-1] + x_edges[1:]) / 2
            y_centers = (y_edges[:-1] + y_edges[1:]) / 2
            if np.any(density):
                ax.contourf(
                    x_centers,
                    y_centers,
                    density.T,
                    levels=8,
                    cmap="Greys",
                    alpha=0.4,
                )
                ax.contour(
                    x_centers,
                    y_centers,
                    density.T,
                    levels=8,
                    colors=MERGED_LABEL_COLORS["train data"],
                    linewidths=0.7,
                    alpha=0.75,
                )

        for label in MERGED_LABEL_ORDER:
            if label == "train data":
                continue
            label_frame = frame[frame["merged_label"] == label]
            if label_frame.empty:
                continue
            for crystal_system in CRYSTAL_SYSTEM_ORDER:
                subset = label_frame[label_frame["crystal_system"] == crystal_system]
                if subset.empty:
                    continue
                ax.scatter(
                    subset["x"],
                    subset["y"],
                    s=40,  # 18,
                    c=MERGED_LABEL_COLORS[label],
                    marker=MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS[crystal_system],
                    alpha=0.78,
                    edgecolors="none",
                )
        ax.set_title(reducer_title(reducer))
        ax.set_xlabel("component 1")
        ax.set_ylabel("component 2")

    category_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=MERGED_LABEL_COLORS[label],
            markeredgecolor="none",
            markersize=7,
            label=label,
        )
        for label in MERGED_LABEL_ORDER
        if label != "train data"
    ]
    crystal_system_handles = [
        plt.Line2D(
            [0],
            [0],
            marker=MATPLOTLIB_CRYSTAL_SYSTEM_MARKERS[crystal_system],
            color="none",
            markerfacecolor="#333333",
            markeredgecolor="none",
            markersize=7,
            label=crystal_system,
        )
        for crystal_system in CRYSTAL_SYSTEM_ORDER
    ]
    fig.legend(
        [*category_handles, *crystal_system_handles],
        [handle.get_label() for handle in [*category_handles, *crystal_system_handles]],
        loc="lower center",
        ncol=5,
        frameon=False,
        bbox_to_anchor=(0.5, -0.08),
    )
    title = f"{model}: {embedding_name} embeddings, merged substitution matches"
    fig.suptitle(title)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def train_projection_kde_bandwidth(train_xy: np.ndarray) -> float:
    dimension = train_xy.shape[1]
    coordinate_scale = float(np.mean(np.std(train_xy, axis=0, ddof=1)))
    if not np.isfinite(coordinate_scale) or coordinate_scale <= 0:
        coordinate_scale = 1.0
    return KDE_SCOTT_FACTOR * coordinate_scale * len(train_xy) ** (-1 / (dimension + 4))


def fit_train_projection_kde(frame: pd.DataFrame) -> KernelDensity | None:
    train_xy = frame.loc[frame["split"] == "train", ["x", "y"]].to_numpy(dtype=float)
    train_xy = train_xy[np.isfinite(train_xy).all(axis=1)]
    if len(train_xy) < KDE_MIN_TRAIN_POINTS:
        return None
    bandwidth = train_projection_kde_bandwidth(train_xy)
    return KernelDensity(kernel="gaussian", bandwidth=bandwidth).fit(train_xy)


def train_kde_likelihood_frame(
    projections: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
) -> pd.DataFrame:
    projections = with_merged_subst_label(projections)
    rows = []
    for reducer in PROJECTION_REDUCERS:
        frame = projections[projections["reducer"] == reducer]
        kde = fit_train_projection_kde(frame)
        if kde is None:
            warnings.warn(
                f"Skipping train KDE for {model}/{embedding_name}/{reducer}: "
                "too few finite train projection points.",
                stacklevel=2,
            )
            continue

        generated = frame[frame["split"] == "generated"].copy()
        generated_xy = generated[["x", "y"]].to_numpy(dtype=float)
        finite_mask = np.isfinite(generated_xy).all(axis=1)
        if not finite_mask.all():
            skipped = int((~finite_mask).sum())
            warnings.warn(
                f"Skipping {skipped} non-finite generated projection points for "
                f"{model}/{embedding_name}/{reducer} train KDE scoring.",
                stacklevel=2,
            )
            generated = generated.loc[finite_mask].copy()
            generated_xy = generated_xy[finite_mask]
        if generated.empty:
            continue

        generated["model"] = model
        generated["embedding_name"] = embedding_name
        generated["train_kde_log_likelihood"] = kde.score_samples(generated_xy)
        rows.append(
            generated[
                [
                    "model",
                    "embedding_name",
                    "reducer",
                    "gen_idx",
                    "label",
                    "merged_label",
                    "composition",
                    "crystal_system",
                    "space_group",
                    "x",
                    "y",
                    "train_kde_log_likelihood",
                ]
            ]
        )

    if not rows:
        return pd.DataFrame(
            columns=[
                "model",
                "embedding_name",
                "reducer",
                "gen_idx",
                "label",
                "merged_label",
                "composition",
                "crystal_system",
                "space_group",
                "x",
                "y",
                "train_kde_log_likelihood",
            ]
        )
    return pd.concat(rows, ignore_index=True)


def likelihood_display_bounds(values: pd.Series) -> tuple[float, float, int, int]:
    finite_values = values[np.isfinite(values)]
    if finite_values.empty:
        return np.nan, np.nan, 0, 0
    lower = float(finite_values.quantile(0.01))
    upper = float(finite_values.quantile(0.99))
    if not np.isfinite(lower) or not np.isfinite(upper) or lower >= upper:
        lower = float(finite_values.min())
        upper = float(finite_values.max())
        if lower == upper:
            padding = max(abs(lower) * 0.01, 1.0)
            lower -= padding
            upper += padding
    low_count = int((finite_values < lower).sum())
    high_count = int((finite_values > upper).sum())
    return lower, upper, low_count, high_count


def can_plot_likelihood_kde(values: pd.Series) -> bool:
    finite_values = values[np.isfinite(values)]
    return len(finite_values) >= 2 and finite_values.nunique() >= 2


def likelihood_kde_bandwidth(values: np.ndarray) -> float:
    value_scale = float(np.std(values, ddof=1))
    if not np.isfinite(value_scale) or value_scale <= 0:
        value_scale = 1.0
    return KDE_SCOTT_FACTOR * value_scale * len(values) ** (-1 / 5)


def likelihood_kde_density(
    values: pd.Series,
    x_grid: np.ndarray,
) -> np.ndarray | None:
    finite_values = values.to_numpy(dtype=float)
    finite_values = finite_values[np.isfinite(finite_values)]
    if len(finite_values) < 2 or len(np.unique(finite_values)) < 2:
        return None

    bandwidth = likelihood_kde_bandwidth(finite_values)
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth).fit(
        finite_values[:, None]
    )
    return np.exp(kde.score_samples(x_grid[:, None]))


def plot_likelihood_median_line(
    ax: plt.Axes,
    values: pd.Series,
    *,
    label: str,
) -> None:
    finite_values = values[np.isfinite(values)]
    if finite_values.empty:
        return
    ax.axvline(
        float(finite_values.median()),
        color=MERGED_LABEL_COLORS[label],
        linestyle="--",
        linewidth=1.5,
        label=f"{label} median only",
    )


def plot_train_kde_likelihood_distribution(
    frame: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
    path: Path | None = None,
) -> None:
    labels = [label for label in MERGED_LABEL_ORDER if label != "train data"]
    present_labels = [
        label for label in labels if label in set(frame["merged_label"].astype(str))
    ]
    fig, axes = plt.subplots(
        1,
        len(PROJECTION_REDUCERS),
        figsize=(10, 4.8),
        constrained_layout=True,
    )
    for ax, reducer in zip(np.ravel(axes), PROJECTION_REDUCERS, strict=True):
        reducer_frame = frame[frame["reducer"] == reducer].copy()
        if reducer_frame.empty or not present_labels:
            ax.set_visible(False)
            continue
        lower, upper, low_count, high_count = likelihood_display_bounds(
            reducer_frame["train_kde_log_likelihood"]
        )
        if not np.isfinite(lower) or not np.isfinite(upper):
            ax.set_visible(False)
            continue
        plot_frame = reducer_frame[
            reducer_frame["train_kde_log_likelihood"].between(lower, upper)
        ]
        kde_rows = []
        for label in present_labels:
            label_values = plot_frame.loc[
                plot_frame["merged_label"] == label,
                "train_kde_log_likelihood",
            ]
            if can_plot_likelihood_kde(label_values):
                kde_rows.append(plot_frame[plot_frame["merged_label"] == label])
            else:
                plot_likelihood_median_line(ax, label_values, label=label)
        if kde_rows:
            kde_frame = pd.concat(kde_rows, ignore_index=True)
            kde_labels = [
                label
                for label in present_labels
                if label in set(kde_frame["merged_label"].astype(str))
            ]
            x_grid = np.linspace(lower, upper, 512)
            total_density = np.zeros_like(x_grid)
            total_count = len(kde_frame)
            for label in kde_labels:
                label_values = kde_frame.loc[
                    kde_frame["merged_label"] == label,
                    "train_kde_log_likelihood",
                ]
                density = likelihood_kde_density(label_values, x_grid)
                if density is None:
                    continue
                scaled_density = density * len(label_values) / total_count
                total_density += scaled_density
                color = MERGED_LABEL_COLORS[label]
                ax.fill_between(x_grid, scaled_density, color=color, alpha=0.25)
                ax.plot(
                    x_grid,
                    scaled_density,
                    color=color,
                    linewidth=1.8,
                    label=label,
                )
            if np.any(total_density):
                ax.plot(
                    x_grid,
                    total_density,
                    color="#111111",
                    linewidth=2.2,
                    label="sum of categories",
                    zorder=5,
                )
        if low_count or high_count:
            ax.text(
                0.02,
                0.96,
                f"clipped: low={low_count}, high={high_count}",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
            )
        ax.set_xlim(lower, upper)
        ax.set_title(reducer_title(reducer))
        ax.set_xlabel("train KDE log-likelihood")
        ax.set_ylabel("density")
        if ax.get_legend_handles_labels()[0]:
            ax.legend(frameon=False, fontsize=8)
    fig.suptitle(f"{model}: {embedding_name} projected train-density likelihood")
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
    display(fig)
    plt.close(fig)


def projection_component_columns(fixed_component: int) -> tuple[str, str]:
    if fixed_component == 1:
        return "x", "y"
    if fixed_component == 2:
        return "y", "x"
    raise ValueError("fixed_component must be 1 or 2")


def contour_density_cross_section(
    frame: pd.DataFrame,
    *,
    fixed_component: int,
    value: float,
) -> tuple[np.ndarray, np.ndarray, float]:
    train = frame[frame["split"] == "train"]
    density, x_edges, y_edges = np.histogram2d(train["x"], train["y"], bins=40)
    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2
    if fixed_component == 1:
        fixed_idx = int(np.abs(x_centers - value).argmin())
        return y_centers, density[fixed_idx, :], float(x_centers[fixed_idx])
    if fixed_component == 2:
        fixed_idx = int(np.abs(y_centers - value).argmin())
        return x_centers, density[:, fixed_idx], float(y_centers[fixed_idx])
    raise ValueError("fixed_component must be 1 or 2")


def kde_density_cross_section(
    frame: pd.DataFrame,
    *,
    fixed_component: int,
    value: float,
    free_values: np.ndarray,
) -> np.ndarray | None:
    kde = fit_train_projection_kde(frame)
    if kde is None:
        return None
    if fixed_component == 1:
        points = np.column_stack([np.full_like(free_values, value), free_values])
    elif fixed_component == 2:
        points = np.column_stack([free_values, np.full_like(free_values, value)])
    else:
        raise ValueError("fixed_component must be 1 or 2")
    return np.exp(kde.score_samples(points))


def cross_section_generated_samples(
    frame: pd.DataFrame,
    *,
    fixed_component: int,
    value: float,
) -> pd.DataFrame:
    fixed_column, _ = projection_component_columns(fixed_component)
    generated = with_merged_subst_label(frame[frame["split"] == "generated"])
    return generated[generated[fixed_column].between(value - 0.5, value + 0.5)]


def cross_section_label_lanes(
    generated: pd.DataFrame,
    *,
    free_column: str,
    x_span: float,
) -> pd.DataFrame:
    if generated.empty:
        return generated.assign(label_lane=pd.Series(dtype=int))

    min_separation = 0.055 * x_span if x_span > 0 else 1.0
    lane_ends: list[float] = []
    lanes = []
    sorted_generated = generated.sort_values(free_column).copy()
    for x_value in sorted_generated[free_column].to_numpy(dtype=float):
        for lane, lane_end in enumerate(lane_ends):
            if x_value - lane_end >= min_separation:
                lane_ends[lane] = x_value
                lanes.append(lane)
                break
        else:
            lane_ends.append(x_value)
            lanes.append(len(lane_ends) - 1)
    return sorted_generated.assign(label_lane=lanes)


def add_cross_section_labeled_rugs(
    ax: plt.Axes,
    generated: pd.DataFrame,
    *,
    fixed_component: int,
    x_span: float,
) -> None:
    _, free_column = projection_component_columns(fixed_component)
    labeled = cross_section_label_lanes(
        generated,
        free_column=free_column,
        x_span=x_span,
    )
    labels = [label for label in MERGED_LABEL_ORDER if label != "train data"]
    for label in labels:
        subset = labeled[labeled["merged_label"] == label]
        if subset.empty:
            continue
        ax.plot(
            subset[free_column],
            np.zeros(len(subset)),
            linestyle="none",
            marker="|",
            markersize=13,
            markeredgewidth=1.6,
            color=MERGED_LABEL_COLORS[label],
            alpha=0.85,
            label=label,
        )


def plot_projection_density_cross_sections(
    projections: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
    reducer: str,
    fixed_component: int,
    value: float,
    path: Path | None = None,
) -> None:
    frame = projections[projections["reducer"] == reducer].copy()
    if frame.empty:
        raise ValueError(f"No projection rows for reducer {reducer!r}")

    fixed_column, free_column = projection_component_columns(fixed_component)
    free_component = 1 if fixed_component == 2 else 2
    free_values, contour_density, nearest_fixed_value = contour_density_cross_section(
        frame,
        fixed_component=fixed_component,
        value=value,
    )
    kde_density = kde_density_cross_section(
        frame,
        fixed_component=fixed_component,
        value=value,
        free_values=free_values,
    )
    generated = cross_section_generated_samples(
        frame,
        fixed_component=fixed_component,
        value=value,
    )

    x_span = float(free_values.max() - free_values.min())
    labeled_generated = cross_section_label_lanes(
        generated,
        free_column=free_column,
        x_span=x_span,
    )
    if labeled_generated.empty:
        label_lane_count = 0
    else:
        label_lane_count = int(labeled_generated["label_lane"].max() + 1)
    bottom_margin = min(0.62, max(0.24, 0.2 + 0.07 * label_lane_count))
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), constrained_layout=False)
    fig.subplots_adjust(bottom=bottom_margin, wspace=0.24)
    axes[0].plot(
        free_values,
        contour_density,
        color=MERGED_LABEL_COLORS["train data"],
        linewidth=1.8,
    )
    axes[0].set_title(
        f"Contour cross section (nearest component {fixed_component}="
        f"{nearest_fixed_value:.3g})"
    )
    axes[0].set_ylabel("binned train density")

    if kde_density is None:
        axes[1].text(0.5, 0.5, "KDE unavailable", ha="center", va="center")
    else:
        axes[1].plot(
            free_values,
            kde_density,
            color=MERGED_LABEL_COLORS["train data"],
            linewidth=1.8,
        )
    axes[1].set_title(f"KDE cross section (component {fixed_component}={value:g})")
    axes[1].set_ylabel("train KDE density")

    for ax in axes:
        add_cross_section_labeled_rugs(
            ax,
            generated,
            fixed_component=fixed_component,
            x_span=x_span,
        )
        ax.set_xlabel(f"component {free_component}")
        ax.set_xlim(float(free_values.min()), float(free_values.max()))
        if generated.empty:
            ax.text(
                0.02,
                0.95,
                "no generated samples in band",
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=8,
            )

    handles = [
        plt.Line2D(
            [0],
            [0],
            linestyle="none",
            marker="|",
            markersize=13,
            markeredgewidth=1.6,
            color=MERGED_LABEL_COLORS[label],
            label=label,
        )
        for label in MERGED_LABEL_ORDER
        if label != "train data"
    ]
    fig.legend(
        handles,
        [handle.get_label() for handle in handles],
        loc="lower center",
        ncol=3,
        frameon=False,
        bbox_to_anchor=(0.5, 0.02),
    )
    fig.suptitle(
        f"{model}: {embedding_name} {reducer_title(reducer)} density cross sections; "
        f"component {fixed_component}={value:g}; "
        f"generated band {value - 0.5:g}<={fixed_column}<={value + 0.5:g}"
    )
    if path is not None:
        fig.savefig(path, bbox_inches="tight")
    display(fig)
    plt.close(fig)


def projection_minimum_distance_frame(
    projections: pd.DataFrame,
    distances: pd.DataFrame,
) -> pd.DataFrame:
    distance_lookup = distances[["gen_idx", "minimum_distance"]]
    train = projections[projections["split"] == "train"].assign(
        minimum_distance=np.nan,
    )
    generated = projections[projections["split"] == "generated"].merge(
        distance_lookup,
        on="gen_idx",
        how="left",
        validate="many_to_one",
    )
    missing_count = int(generated["minimum_distance"].isna().sum())
    if missing_count:
        warnings.warn(
            f"{missing_count} projected generated rows are missing nearest-train "
            "distances and will be omitted from distance-colored plots.",
            stacklevel=2,
        )
        generated = generated.dropna(subset=["minimum_distance"])
    return pd.concat([train, generated], ignore_index=True)


def minimum_distance_color_bound(projections: pd.DataFrame) -> float:
    distances = projections.loc[
        projections["split"] == "generated",
        "minimum_distance",
    ].dropna()
    if distances.empty:
        return 1.0
    upper = float(distances.quantile(0.99))
    if not np.isfinite(upper) or upper <= 0:
        return float(distances.max()) or 1.0
    return upper


def plot_minimum_distance_projection_grid(
    projections: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
) -> go.Figure:
    cmax = minimum_distance_color_bound(projections)
    fig = make_subplots(
        rows=1,
        cols=len(PROJECTION_REDUCERS),
        subplot_titles=[reducer_title(reducer) for reducer in PROJECTION_REDUCERS],
        horizontal_spacing=0.06,
    )

    for col, reducer in enumerate(PROJECTION_REDUCERS, start=1):
        frame = projections[projections["reducer"] == reducer]
        train = frame[frame["split"] == "train"]
        if not train.empty:
            fig.add_trace(
                go.Histogram2dContour(
                    x=train["x"],
                    y=train["y"],
                    colorscale=[
                        [0, rgba(LABEL_COLORS["train data"], 0.0)],
                        [1, rgba(LABEL_COLORS["train data"], 0.65)],
                    ],
                    contours={"coloring": "fill", "showlines": True},
                    line={"color": rgba(LABEL_COLORS["train data"], 0.8), "width": 1},
                    opacity=0.55,
                    showscale=False,
                    name="train data density",
                    legendgroup="train data",
                    showlegend=col == 1,
                    hoverinfo="skip",
                ),
                row=1,
                col=col,
            )

        generated = frame[frame["split"] == "generated"].sort_values(
            "minimum_distance",
            ascending=False,
        )
        if generated.empty:
            continue
        fig.add_trace(
            go.Scattergl(
                x=generated["x"],
                y=generated["y"],
                mode="markers",
                marker={
                    "size": 7,
                    "color": generated["minimum_distance"],
                    "colorscale": "Viridis",
                    "cmin": 0,
                    "cmax": cmax,
                    "opacity": 0.86,
                    "line": {"width": 0},
                    "showscale": col == len(PROJECTION_REDUCERS),
                    "colorbar": {"title": "nearest train distance"},
                },
                name="generated structures",
                legendgroup="generated structures",
                showlegend=col == 1,
                customdata=np.stack(
                    [
                        generated["composition"].astype(str),
                        generated["crystal_system"].astype(str),
                        generated["space_group"].astype(str),
                        generated["gen_idx"].astype(str),
                        generated["label"].astype(str),
                        generated["model"].astype(str),
                        generated["minimum_distance"].map("{:.4g}".format),
                    ],
                    axis=-1,
                ),
                hovertemplate=(
                    "composition: %{customdata[0]}<br>"
                    "crystal system: %{customdata[1]}<br>"
                    "space group: %{customdata[2]}<br>"
                    "gen idx: %{customdata[3]}<br>"
                    "category: %{customdata[4]}<br>"
                    "model: %{customdata[5]}<br>"
                    "nearest train distance: %{customdata[6]}"
                    "<extra></extra>"
                ),
            ),
            row=1,
            col=col,
        )

        fig.update_xaxes(title_text="component 1", row=1, col=col)
        fig.update_yaxes(title_text="component 2", row=1, col=col)

    title = f"{model}: {embedding_name} embeddings by nearest train distance"
    fig.update_layout(
        title=title,
        template="plotly_white",
        width=1200,
        height=430,
        legend={"orientation": "h", "y": -0.22, "x": 0.5, "xanchor": "center"},
        margin={"l": 55, "r": 95, "t": 70, "b": 95},
    )
    return fig


def save_minimum_distance_projection_pdf(
    projections: pd.DataFrame,
    *,
    model: str,
    embedding_name: str,
    path: Path,
) -> None:
    cmax = minimum_distance_color_bound(projections)
    fig, axes = plt.subplots(
        1,
        len(PROJECTION_REDUCERS),
        figsize=(5 * len(PROJECTION_REDUCERS), 4.8),
        constrained_layout=True,
    )
    scatter = None
    for ax, reducer in zip(axes, PROJECTION_REDUCERS, strict=True):
        frame = projections[projections["reducer"] == reducer]
        train = frame[frame["split"] == "train"]
        if not train.empty:
            density, x_edges, y_edges = np.histogram2d(train["x"], train["y"], bins=40)
            x_centers = (x_edges[:-1] + x_edges[1:]) / 2
            y_centers = (y_edges[:-1] + y_edges[1:]) / 2
            if np.any(density):
                ax.contourf(
                    x_centers,
                    y_centers,
                    density.T,
                    levels=8,
                    cmap="Greys",
                    alpha=0.4,
                )
                ax.contour(
                    x_centers,
                    y_centers,
                    density.T,
                    levels=8,
                    colors=LABEL_COLORS["train data"],
                    linewidths=0.7,
                    alpha=0.75,
                )

        generated = frame[frame["split"] == "generated"].sort_values(
            "minimum_distance",
            ascending=False,
        )
        if not generated.empty:
            scatter = ax.scatter(
                generated["x"],
                generated["y"],
                s=18,
                c=generated["minimum_distance"],
                cmap="viridis",
                vmin=0,
                vmax=cmax,
                alpha=0.86,
                edgecolors="none",
            )
        ax.set_title(reducer_title(reducer))
        ax.set_xlabel("component 1")
        ax.set_ylabel("component 2")

    if scatter is not None:
        colorbar = fig.colorbar(scatter, ax=axes, shrink=0.86, pad=0.02)
        colorbar.set_label("nearest train distance")
    title = f"{model}: {embedding_name} embeddings by nearest train distance"
    fig.suptitle(title)
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


cross_section_projections = projection_frame(
    CROSS_SECTION_MODEL,
    CROSS_SECTION_EMBEDDING,
)
plot_projection_density_cross_sections(
    cross_section_projections,
    model=CROSS_SECTION_MODEL,
    embedding_name=CROSS_SECTION_EMBEDDING,
    reducer=CROSS_SECTION_REDUCER,
    fixed_component=CROSS_SECTION_FIXED_COMPONENT,
    value=CROSS_SECTION_VALUE,
    path=FIGURE_DIR
    / (
        f"{CROSS_SECTION_MODEL}_{CROSS_SECTION_EMBEDDING}_{CROSS_SECTION_REDUCER}"
        f"_component{CROSS_SECTION_FIXED_COMPONENT}_{CROSS_SECTION_VALUE:g}"
        "_density_cross_sections.pdf"
    ),
)


train_kde_likelihood_tables: dict[tuple[str, str], pd.DataFrame] = {}
for model in complete_models:
    for embedding_name in EMBEDDING_SETTINGS:
        projections = likelihood_projection_frame(model, embedding_name)
        likelihood_frame = train_kde_likelihood_frame(
            projections,
            model=model,
            embedding_name=embedding_name,
        )
        train_kde_likelihood_tables[(model, embedding_name)] = likelihood_frame
        if likelihood_frame.empty:
            continue
        stem = f"{model}_{embedding_name}_train_kde_log_likelihood"
        plot_train_kde_likelihood_distribution(
            likelihood_frame,
            model=model,
            embedding_name=embedding_name,
            path=FIGURE_DIR / f"{stem}.pdf",
        )

if train_kde_likelihood_tables:
    train_kde_likelihood_summary = (
        pd.concat(train_kde_likelihood_tables.values(), ignore_index=True)
        .groupby(
            ["model", "embedding_name", "reducer", "merged_label"],
            observed=False,
        )["train_kde_log_likelihood"]
        .describe()
    )
else:
    train_kde_likelihood_summary = pd.DataFrame()
display(train_kde_likelihood_summary)


for model in complete_models:
    for embedding_name in EMBEDDING_SETTINGS:
        projections = projection_frame(model, embedding_name)
        distance_projections = projection_minimum_distance_frame(
            projections,
            minimum_distance_frame(model, embedding_name),
        )
        stem = f"{model}_{embedding_name}_projections"

        distance_fig = plot_minimum_distance_projection_grid(
            distance_projections,
            model=model,
            embedding_name=embedding_name,
        )
        save_minimum_distance_projection_pdf(
            distance_projections,
            model=model,
            embedding_name=embedding_name,
            path=FIGURE_DIR / f"{stem}_nearest_train_distance.pdf",
        )
        display(distance_fig)

        merged_fig = plot_merged_subst_projection_grid(
            projections,
            model=model,
            embedding_name=embedding_name,
        )
        save_merged_subst_projection_pdf(
            projections,
            model=model,
            embedding_name=embedding_name,
            path=FIGURE_DIR / f"{stem}_merged_subst_crystal_system.pdf",
        )
        display(merged_fig)

## Projection diagnostics

Quantify PCA component contribution and generated-to-train neighbor preservation for PCA and UMAP projections.


In [ ]:
QUALITY_K_VALUES = (5, 10, 20)
QUALITY_REDUCERS = PROJECTION_REDUCERS
QUALITY_METRICS = ("trustworthiness", "continuity", "knn_preservation")


def projection_quality_cache_path(
    model: str,
    embedding_name: str,
    reducer: str,
    k_values: tuple[int, ...],
) -> Path:
    metric = embedding_distance_metric(embedding_name)
    k_label = "-".join(str(k) for k in k_values)
    seed_label = projection_seed_label(
        reducer,
        MAX_PLOT_POINTS_PER_GENERATED_CATEGORY,
    )
    name = (
        f"{model}_{embedding_name}_{reducer}"
        f"_generated_train_metric={metric}"
        f"_f={FILTER_GENERATED_TO_METASTABLE_SMACT_VALID}"
        f"_n={MAX_PLOT_POINTS_PER_GENERATED_CATEGORY}"
        f"_k={k_label}"
        f"{seed_label}"
        ".pkl.gz"
    )
    return CACHE_DIR / "quality_metrics" / name


def selected_scaled_embedding_data(
    model: str,
    embedding_name: str,
) -> tuple[np.ndarray, pd.DataFrame]:
    metadata = embedding_tables[(model, embedding_name)]
    embeddings = embedding_arrays[(model, embedding_name)]
    train_indices = metadata.index[metadata["split"] == "train"].to_numpy()
    train_metadata = metadata.iloc[train_indices].reset_index(drop=True)
    scaled_embeddings = StandardScaler().fit_transform(embeddings[train_indices])
    return scaled_embeddings, train_metadata


def pca_component_contribution_frame() -> pd.DataFrame:
    rows = []
    for model in complete_models:
        for embedding_name in EMBEDDING_SETTINGS:
            scaled_embeddings, selected_metadata = selected_scaled_embedding_data(
                model,
                embedding_name,
            )
            pca = PCA(**REDUCER_SETTINGS["pca"]).fit(scaled_embeddings)
            ratios = pca.explained_variance_ratio_
            rows.append(
                {
                    "model": model,
                    "embedding_name": embedding_name,
                    "n_points": len(selected_metadata),
                    "component_1_pct": float(100 * ratios[0]),
                    "component_2_pct": float(100 * ratios[1]),
                    "total_pct": float(100 * ratios[:2].sum()),
                }
            )
    return pd.DataFrame(rows)


def generated_position_by_gen_idx(model: str, embedding_name: str) -> dict[int, int]:
    metadata = embedding_tables[(model, embedding_name)]
    generated = metadata[metadata["split"] == "generated"].reset_index(drop=True)
    return {
        int(gen_idx): position
        for position, gen_idx in enumerate(generated["gen_idx"].astype(int))
    }


def selected_generated_positions(
    model: str,
    embedding_name: str,
    generated: pd.DataFrame,
) -> np.ndarray:
    position_by_gen_idx = generated_position_by_gen_idx(model, embedding_name)
    positions = generated["gen_idx"].astype(int).map(position_by_gen_idx)
    if positions.isna().any():
        missing = generated.loc[positions.isna(), "gen_idx"].astype(str).tolist()
        raise ValueError(
            f"Could not align generated indices for {model}/{embedding_name}: "
            f"{missing[:5]}"
        )
    return positions.to_numpy(dtype=int)


def generated_to_train_embedding_distances(
    *,
    model: str,
    embedding_name: str,
    generated: pd.DataFrame,
    train: pd.DataFrame,
) -> np.ndarray:
    distances = distance_matrix(model, embedding_name)
    generated_positions = selected_generated_positions(model, embedding_name, generated)
    train_indices = train["structure_idx"].astype(int).to_numpy()
    return distances[np.ix_(generated_positions, train_indices)]


def generated_to_train_projection_distances(
    generated: pd.DataFrame,
    train: pd.DataFrame,
) -> np.ndarray:
    generated_xy = generated[["x", "y"]].to_numpy(dtype=float)
    train_xy = train[["x", "y"]].to_numpy(dtype=float)
    chunks = [
        np.asarray(chunk, dtype=np.float32)
        for chunk in pairwise_distances_chunked(generated_xy, train_xy)
    ]
    return np.vstack(chunks)


def neighbor_orders_and_ranks(distances: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    order = np.argsort(distances, axis=1)
    ranks = np.empty(order.shape, dtype=np.int32)
    row_indices = np.arange(order.shape[0])[:, None]
    ranks[row_indices, order] = np.arange(1, order.shape[1] + 1, dtype=np.int32)
    return order, ranks


def bounded_unit_score(score: float) -> float:
    return float(np.clip(score, 0.0, 1.0))


def generated_to_train_rank_metrics(
    *,
    embedding_order: np.ndarray,
    embedding_ranks: np.ndarray,
    projection_order: np.ndarray,
    projection_ranks: np.ndarray,
    k: int,
) -> dict[str, float]:
    if embedding_order.shape != projection_order.shape:
        raise ValueError(
            "Embedding and projection neighbor orders have mismatched shapes: "
            f"{embedding_order.shape} vs {projection_order.shape}."
        )
    n_generated, n_train = embedding_order.shape
    if n_generated == 0:
        raise ValueError("At least one generated point is required.")
    if not 1 <= k < n_train:
        raise ValueError(f"k must satisfy 1 <= k < {n_train}; got {k}.")

    embedding_top = embedding_order[:, :k]
    projection_top = projection_order[:, :k]

    trust_penalty = 0.0
    continuity_penalty = 0.0
    preserved_fraction = 0.0
    for row_idx in range(n_generated):
        embedding_set = set(int(idx) for idx in embedding_top[row_idx])
        projection_set = set(int(idx) for idx in projection_top[row_idx])
        preserved_fraction += len(embedding_set & projection_set) / k
        trust_penalty += sum(
            float(embedding_ranks[row_idx, idx] - k)
            for idx in projection_set - embedding_set
        )
        continuity_penalty += sum(
            float(projection_ranks[row_idx, idx] - k)
            for idx in embedding_set - projection_set
        )

    normalizer = 2.0 / (n_generated * k * (2 * n_train - 3 * k - 1))
    return {
        "trustworthiness": bounded_unit_score(1.0 - normalizer * trust_penalty),
        "continuity": bounded_unit_score(1.0 - normalizer * continuity_penalty),
        "knn_preservation": bounded_unit_score(preserved_fraction / n_generated),
    }


def projection_quality_metric_frame(
    model: str,
    embedding_name: str,
    reducer: str,
    *,
    k_values: tuple[int, ...] = QUALITY_K_VALUES,
) -> pd.DataFrame:
    cache_path = projection_quality_cache_path(
        model,
        embedding_name,
        reducer,
        k_values,
    )
    cached = load_cache(cache_path)
    if cached is not None:
        return pd.DataFrame(cached["metrics"])

    projections = projection_frame(model, embedding_name)
    frame = projections[projections["reducer"] == reducer].copy()
    train = frame[frame["split"] == "train"].copy()
    generated = frame[frame["split"] == "generated"].copy()
    if train.empty or generated.empty:
        metrics = pd.DataFrame(
            columns=[
                "model",
                "embedding_name",
                "reducer",
                "k",
                "n_generated",
                "n_train",
                *QUALITY_METRICS,
            ]
        )
        save_cache(cache_path, {"metrics": metrics, "metadata": {}})
        return metrics

    embedding_distances = generated_to_train_embedding_distances(
        model=model,
        embedding_name=embedding_name,
        generated=generated,
        train=train,
    )
    projection_distances = generated_to_train_projection_distances(generated, train)

    embedding_order, embedding_ranks = neighbor_orders_and_ranks(embedding_distances)
    projection_order, projection_ranks = neighbor_orders_and_ranks(projection_distances)

    rows = []
    for k in k_values:
        metrics = generated_to_train_rank_metrics(
            embedding_order=embedding_order,
            embedding_ranks=embedding_ranks,
            projection_order=projection_order,
            projection_ranks=projection_ranks,
            k=k,
        )
        rows.append(
            {
                "model": model,
                "embedding_name": embedding_name,
                "reducer": reducer,
                "k": k,
                "n_generated": len(generated),
                "n_train": len(train),
                **metrics,
            }
        )

    metrics = pd.DataFrame(rows)
    save_cache(
        cache_path,
        {
            "metadata": {
                "model": model,
                "embedding_name": embedding_name,
                "reducer": reducer,
                "k_values": k_values,
                "metric": embedding_distance_metric(embedding_name),
                "n_generated": len(generated),
                "n_train": len(train),
                "scope": "generated-to-train",
            },
            "created_at": datetime.now(UTC).isoformat(),
            "metrics": metrics,
        },
    )
    return metrics


def projection_quality_metrics_frame() -> pd.DataFrame:
    tables = []
    for model in complete_models:
        for embedding_name in EMBEDDING_SETTINGS:
            for reducer in QUALITY_REDUCERS:
                tables.append(
                    projection_quality_metric_frame(
                        model,
                        embedding_name,
                        reducer,
                    )
                )
    if not tables:
        return pd.DataFrame()
    return pd.concat(tables, ignore_index=True)


def plot_projection_quality_metrics(
    frame: pd.DataFrame,
    *,
    path: Path | None = None,
) -> None:
    if frame.empty:
        display(Markdown("No projection quality metrics to plot."))
        return

    plot_frame = frame.melt(
        id_vars=["model", "embedding_name", "reducer", "k"],
        value_vars=list(QUALITY_METRICS),
        var_name="metric",
        value_name="score",
    )
    plot_frame["model_embedding"] = (
        plot_frame["model"].astype(str)
        + " / "
        + plot_frame["embedding_name"].astype(str)
    )
    grid = sns.relplot(
        data=plot_frame,
        x="k",
        y="score",
        hue="metric",
        hue_order=list(QUALITY_METRICS),
        row="model_embedding",
        col="reducer",
        col_order=list(QUALITY_REDUCERS),
        kind="line",
        marker="o",
        height=2.5,
        aspect=1.35,
        facet_kws={"sharex": True, "sharey": True},
    )
    grid.set_axis_labels("k", "score")
    grid.set_titles(row_template="{row_name}", col_template="{col_name}")
    for ax in grid.axes.flat:
        ax.set_ylim(0.0, 1.02)
        ax.set_xticks(list(QUALITY_K_VALUES))
        ax.grid(True, axis="y", alpha=0.25)
    grid.figure.suptitle("Generated-to-train projection quality", y=1.01)
    if path is not None:
        grid.figure.savefig(path, bbox_inches="tight")
    display(grid.figure)
    plt.close(grid.figure)


pca_component_contributions = pca_component_contribution_frame()
display(
    pca_component_contributions.style.format(
        {
            "component_1_pct": "{:.2f}",
            "component_2_pct": "{:.2f}",
            "total_pct": "{:.2f}",
        }
    )
)

projection_quality_metrics = projection_quality_metrics_frame()
display(
    projection_quality_metrics.style.format(
        {
            "trustworthiness": "{:.4f}",
            "continuity": "{:.4f}",
            "knn_preservation": "{:.4f}",
        }
    )
)
plot_projection_quality_metrics(
    projection_quality_metrics,
    path=FIGURE_DIR / "projection_quality_metrics.pdf",
)

In [ ]:
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

NEIGHBOR_VIEW_MODEL = "mattergen"
NEIGHBOR_VIEW_EMBEDDING = "amd"
NEIGHBOR_VIEW_GEN_IDX = 8103
NEIGHBOR_COUNT = 3


def conventional_structure(structure: Any) -> Any:
    return SpacegroupAnalyzer(structure).get_conventional_standard_structure()


def lattice_parameter_text(structure: Any) -> str:
    lattice = structure.lattice
    return (
        f"a={lattice.a:.1f}, b={lattice.b:.1f}, c={lattice.c:.1f}; "
        f"alpha={lattice.alpha:.0f}, beta={lattice.beta:.0f}, gamma={lattice.gamma:.0f}"
    )


def structure_metadata(structure: Any) -> dict[str, str]:
    analyzer = SpacegroupAnalyzer(structure)
    space_group_symbol, space_group_number = (
        analyzer.get_space_group_symbol(),
        analyzer.get_space_group_number(),
    )
    return {
        "composition": structure.composition.reduced_formula,
        "space_group": f"{space_group_symbol} ({space_group_number})",
        "crystal_system": crystal_system_from_spg_num(space_group_number),
        "lattice_parameters": lattice_parameter_text(structure),
    }


def validate_neighbor_selection(model: str, embedding_name: str, count: int) -> None:
    if model not in complete_models:
        raise ValueError(f"Unknown model: {model!r}")
    if embedding_name not in EMBEDDING_SETTINGS:
        raise ValueError(f"Unknown embedding: {embedding_name!r}")
    if count < 1:
        raise ValueError("NEIGHBOR_COUNT must be at least 1.")
    if len(train_structures) < count:
        raise ValueError(
            f"Requested {count} neighbors, but only "
            f"{len(train_structures)} train structures exist."
        )


def selected_generated_row(model: str, embedding_name: str, gen_idx: int) -> pd.Series:
    validate_neighbor_selection(model, embedding_name, count=1)
    metadata = embedding_tables[(model, embedding_name)]
    generated = metadata[metadata["split"] == "generated"].reset_index(drop=True)
    matches = generated[generated["gen_idx"].astype(int) == int(gen_idx)]
    if matches.empty:
        raise ValueError(
            f"No generated sample with gen_idx={gen_idx} for {model}/{embedding_name}."
        )
    if len(matches) > 1:
        raise ValueError(
            f"Found multiple generated samples with gen_idx={gen_idx} "
            f"for {model}/{embedding_name}."
        )
    return matches.iloc[0]


def neighbor_rows(
    model: str, embedding_name: str, gen_idx: int, count: int
) -> pd.DataFrame:
    validate_neighbor_selection(model, embedding_name, count)
    generated_row = selected_generated_row(model, embedding_name, gen_idx)
    distances = distance_matrix(model, embedding_name)
    generated_position = int(generated_row.name)
    nearest_train_indices = np.argsort(distances[generated_position])[:count]
    nearest_distances = distances[generated_position, nearest_train_indices]

    return train_metadata.iloc[nearest_train_indices].assign(
        train_idx=nearest_train_indices.astype(int),
        distance=nearest_distances.astype(float),
    )


def generated_panel_title(model: str, gen_idx: int) -> str:
    return f"Generated sample<br>model={model}; gen_idx={gen_idx}"


def train_panel_title(rank: int, train_idx: int) -> str:
    return f"Neighbor {rank}<br>train_idx={train_idx}"


def safe_label(value: str) -> str:
    return "".join(char if char.isalnum() else "_" for char in value)


def generated_cif_path(
    model: str, embedding_name: str, gen_idx: int, structure: Any
) -> Path:
    formula = safe_label(structure.composition.reduced_formula)
    return FIGURE_DIR / (
        f"{model}_{embedding_name}_gen_{gen_idx}_{formula}_conventional.cif"
    )


def neighbor_cif_path(
    model: str,
    embedding_name: str,
    gen_idx: int,
    rank: int,
    train_idx: int,
    structure: Any,
) -> Path:
    formula = safe_label(structure.composition.reduced_formula)
    return FIGURE_DIR / (
        f"{model}_{embedding_name}_gen_{gen_idx}_neighbor_{rank}_"
        f"train_{train_idx}_{formula}_conventional.cif"
    )


def save_structure_cif(structure: Any, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    structure.to(filename=path)
    return path


def generated_metadata_frame(
    model: str,
    embedding_name: str,
    gen_idx: int,
    structure: Any,
) -> pd.DataFrame:
    metadata = structure_metadata(structure)
    return pd.DataFrame(
        [
            {
                "model": model,
                "embedding_name": embedding_name,
                "metric": embedding_distance_metric(embedding_name),
                "gen_idx": gen_idx,
                "composition": metadata["composition"],
                "space_group": metadata["space_group"],
                "crystal_system": metadata["crystal_system"],
                "lattice_parameters": metadata["lattice_parameters"],
            }
        ]
    )


def neighbor_metadata_frame(neighbor_frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for rank, row in enumerate(neighbor_frame.itertuples(index=False), start=1):
        metadata = structure_metadata(
            conventional_structure(train_structures[int(row.train_idx)])
        )
        rows.append(
            {
                "neighbor_rank": rank,
                "train_idx": int(row.train_idx),
                "embedding_distance": float(row.distance),
                "composition": metadata["composition"],
                "space_group": metadata["space_group"],
                "crystal_system": metadata["crystal_system"],
                "lattice_parameters": metadata["lattice_parameters"],
            }
        )
    return pd.DataFrame(rows)


neighbor_generated_row = selected_generated_row(
    NEIGHBOR_VIEW_MODEL,
    NEIGHBOR_VIEW_EMBEDDING,
    NEIGHBOR_VIEW_GEN_IDX,
)
neighbor_generated_structures = maybe_limit_structures(
    load_pickle_gz(model_required_paths(NEIGHBOR_VIEW_MODEL)["generated_structures"])
)
neighbor_generated_structure = conventional_structure(
    neighbor_generated_structures[int(neighbor_generated_row["gen_idx"])]
)
neighbor_frame = neighbor_rows(
    NEIGHBOR_VIEW_MODEL,
    NEIGHBOR_VIEW_EMBEDDING,
    NEIGHBOR_VIEW_GEN_IDX,
    NEIGHBOR_COUNT,
)

display(
    Markdown(
        f"## Generated sample and nearest train neighbors  \n"
        f"model=`{NEIGHBOR_VIEW_MODEL}`; embedding=`{NEIGHBOR_VIEW_EMBEDDING}`; "
        f"metric=`{embedding_distance_metric(NEIGHBOR_VIEW_EMBEDDING)}`; "
        f"gen_idx=`{NEIGHBOR_VIEW_GEN_IDX}`"
    )
)

generated_metadata = generated_metadata_frame(
    NEIGHBOR_VIEW_MODEL,
    NEIGHBOR_VIEW_EMBEDDING,
    NEIGHBOR_VIEW_GEN_IDX,
    neighbor_generated_structure,
)
neighbor_metadata = neighbor_metadata_frame(neighbor_frame)

generated_fig = structure_2d(
    {"generated": neighbor_generated_structure},
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _structure, _key: generated_panel_title(
        NEIGHBOR_VIEW_MODEL,
        NEIGHBOR_VIEW_GEN_IDX,
    ),
)
generated_fig.update_layout(height=360, margin={"l": 10, "r": 10, "t": 80, "b": 10})
display(generated_fig)
display(Markdown("### Generated sample metadata"))
display(generated_metadata)

neighbor_structures = {
    f"train_{int(row.train_idx)}": conventional_structure(
        train_structures[int(row.train_idx)]
    )
    for row in neighbor_frame.itertuples(index=False)
}
neighbor_titles = {
    f"train_{int(row.train_idx)}": train_panel_title(
        rank,
        int(row.train_idx),
    )
    for rank, row in enumerate(neighbor_frame.itertuples(index=False), start=1)
}
neighbor_fig = structure_2d(
    neighbor_structures,
    n_cols=NEIGHBOR_COUNT,
    show_cell=True,
    standardize_struct=False,
    subplot_title=lambda _structure, key: neighbor_titles[key],
)
neighbor_fig.update_layout(height=420, margin={"l": 10, "r": 10, "t": 80, "b": 10})
display(Markdown("### Nearest train structures"))
display(neighbor_fig)
display(Markdown("### Nearest train metadata"))
display(neighbor_metadata)

generated_cif = save_structure_cif(
    neighbor_generated_structure,
    generated_cif_path(
        NEIGHBOR_VIEW_MODEL,
        NEIGHBOR_VIEW_EMBEDDING,
        NEIGHBOR_VIEW_GEN_IDX,
        neighbor_generated_structure,
    ),
)
neighbor_cifs = [
    save_structure_cif(
        structure,
        neighbor_cif_path(
            NEIGHBOR_VIEW_MODEL,
            NEIGHBOR_VIEW_EMBEDDING,
            NEIGHBOR_VIEW_GEN_IDX,
            rank,
            int(key.removeprefix("train_")),
            structure,
        ),
    )
    for rank, (key, structure) in enumerate(neighbor_structures.items(), start=1)
]
display(
    Markdown(
        "### Wrote CIF files  \n"
        + f"- `{generated_cif}`  \n"
        + "".join(f"- `{path}`  \n" for path in neighbor_cifs)
    )
)